In [5]:
# Import libraries
import pandas as pd
import datetime as dt
import numpy as np
import os

# Upload instruments and their characteristics
InitData = pd.read_excel('M:/SHARE/Risk_Invest/Рыночные риски/Исходные файлы для отчетов/Monthly BPV Report/Выгрузка.xlsx')

# Insert report date
Report_Date = pd.to_datetime(InitData['REPORT_DATE'].iloc[0], format = '%d.%m.%Y')
 
appended = pd.DataFrame()
    
for i in range (0, len(InitData['INSTRUMENT_NAME'])):

    # Construct datelist
    datelist = pd.date_range(pd.to_datetime(Report_Date, format = '%d.%m.%Y'), periods = 365).tolist()
    df = pd.DataFrame(datelist, columns = ['DATE'])
    df.head()

    # Cracteristics of the bond
    df['Report_Date'] = Report_Date
    df['Instrument_Name'] = InitData['INSTRUMENT_NAME'].iloc[i]
    df['BOOK'] = InitData['BOOK'].iloc[i]
    df['Nearest_Offer'] = pd.to_datetime(InitData['NEAREST_OFFER_CORRECT'].iloc[i], format = '%d.%m.%Y')
    df['2nd_Nearest_Offer'] = pd.to_datetime(InitData['SECOND_NEAREST_OFFER_CORRECT'].iloc[i], format = '%d.%m.%Y')
    df['3rd_Nearest_Offer'] = pd.to_datetime(InitData['THIRD_NEAREST_OFFER_CORRECT'].iloc[i], format = '%d.%m.%Y')
    df['Nearest_Coupon'] = pd.to_datetime(InitData['NEAREST_COUPON_CORRECT'].iloc[i], format = '%d.%m.%Y')
    df['2nd_Nearest_Coupon'] = pd.to_datetime(InitData['SECOND_NEAREST_COUPON_CORRECT'].iloc[i], format = '%d.%m.%Y')
    df['3rd_Nearest_Coupon'] = pd.to_datetime(InitData['THIRD_NEAREST_COUPON_CORRECT'].iloc[i], format = '%d.%m.%Y')

    df['AI_One_Day'] = InitData['AI_ONE_DAY'].iloc[i]
    df['Market_Price'] = InitData['MARKET_PRICE'].iloc[i]
    df['Quantity'] = InitData['QUANTITY'].iloc[i]
    df['Nominal'] = InitData['NOMINAL'].iloc[i]
    df['Coupon_Period'] = InitData['COUPON_PERIOD'].iloc[i]

    df['AI_0'] = np.where(df['DATE'] < df['Nearest_Coupon'], InitData['AI_0'].iloc[i], 0)
    df['Full_MD'] = (df['2nd_Nearest_Offer'] - df['Nearest_Offer']) / 365
    df['Full_MD'] = (df['Full_MD'] / np.timedelta64(1, 'D')).astype(float)
    df['Initial_MD'] = np.where(df['DATE'] < df['Nearest_Offer'], InitData['INITIAL_MD'].iloc[i], df['Full_MD'])

    # Adding extra columns for offers and coupons
    df['TRUE_Nearest_Offer'] = np.where((df['DATE'] < df['Nearest_Offer']), df['Nearest_Offer'].iloc[0], np.where((df['DATE'] >= df['Nearest_Offer']) & (df['DATE'] < df['2nd_Nearest_Offer']), df['2nd_Nearest_Offer'].iloc[0], df['3rd_Nearest_Offer'].iloc[0]))
    df['TRUE_Nearest_Coupon'] = np.where((df['DATE'] < df['Nearest_Coupon']), df['Nearest_Coupon'].iloc[0], np.where((df['DATE'] >= df['Nearest_Coupon']) & (df['DATE'] < df['2nd_Nearest_Coupon']), df['2nd_Nearest_Coupon'].iloc[0], df['3rd_Nearest_Coupon'].iloc[0]))

    # Computing times in datetime type
    df['Days_Since_Report_to_True_Offer'] = np.where(df['DATE'] < df['Nearest_Offer'], df['Nearest_Offer'] - df['Report_Date'], np.where((df['DATE'] < df['2nd_Nearest_Offer']) & (df['DATE'] >= df['Nearest_Offer']), df['2nd_Nearest_Offer'] - df['Nearest_Offer'], np.where((df['DATE'] < df['3rd_Nearest_Offer']) & (df['DATE'] >= df['2nd_Nearest_Offer']), df['3rd_Nearest_Offer'] - df['2nd_Nearest_Offer'], df['DATE'] - df['DATE'])))
    df['Days_Since_Report_to_True_Coupon'] = np.where(df['DATE'] < df['Nearest_Coupon'], df['Nearest_Coupon'] - df['Report_Date'], np.where((df['DATE'] < df['2nd_Nearest_Coupon']) & (df['DATE'] >= df['Nearest_Coupon']), df['2nd_Nearest_Coupon'] - df['Nearest_Coupon'], np.where((df['DATE'] < df['3rd_Nearest_Coupon']) & (df['DATE'] >= df['2nd_Nearest_Coupon']), df['3rd_Nearest_Coupon'] - df['2nd_Nearest_Coupon'], df['DATE'] - df['DATE'])))
    df['Days_Since_Report_to_Date'] = df['DATE'] - df['Report_Date']
    df['Days_Since_Report_to_Date_Coupon'] = np.where(df['DATE'] < df['Nearest_Coupon'], df['DATE'] - df['Report_Date'], np.where((df['DATE'] < df['2nd_Nearest_Coupon']) & (df['DATE'] >= df['Nearest_Coupon']), df['DATE'] - df['Nearest_Coupon'], np.where((df['DATE'] < df['3rd_Nearest_Coupon']) & (df['DATE'] >= df['2nd_Nearest_Coupon']), df['DATE'] - df['2nd_Nearest_Coupon'], np.where(df['DATE'] >= df['3rd_Nearest_Coupon'], df['DATE'] - df['3rd_Nearest_Coupon'], df['DATE'] - df['DATE']))))
    df['Days_Since_Report_to_Date_Offer'] = np.where(df['DATE'] < df['Nearest_Offer'], df['DATE'] - df['Report_Date'], np.where((df['DATE'] < df['2nd_Nearest_Offer']) & (df['DATE'] >= df['Nearest_Offer']), df['DATE'] - df['Nearest_Offer'], np.where((df['DATE'] < df['3rd_Nearest_Offer']) & (df['DATE'] >= df['2nd_Nearest_Offer']),  df['DATE'] - df['2nd_Nearest_Offer'], np.where(df['DATE'] >= df['3rd_Nearest_Offer'], df['DATE'] - df['3rd_Nearest_Offer'], df['DATE'] - df['DATE']))))

    # Convert datetime to integers
    df['Days_Since_Report_to_True_Offer'] = (df['Days_Since_Report_to_True_Offer'] / np.timedelta64(1, 'D')).astype(int)
    df['Days_Since_Report_to_True_Coupon'] = (df['Days_Since_Report_to_True_Coupon'] / np.timedelta64(1, 'D')).astype(int)
    df['Days_Since_Report_to_Date'] = (df['Days_Since_Report_to_Date'] / np.timedelta64(1, 'D')).astype(int)
    df['Days_Since_Report_to_Date_Coupon'] = (df['Days_Since_Report_to_Date_Coupon'] / np.timedelta64(1, 'D')).astype(int)
    df['Days_Since_Report_to_Date_Offer'] = (df['Days_Since_Report_to_Date_Offer'] / np.timedelta64(1, 'D')).astype(int)

    # Computing daily AI and MD 
    df['AI'] = df['AI_0'] + df['AI_One_Day'] * df['Days_Since_Report_to_Date_Coupon']
    df['MD_on_Date'] = df['Initial_MD'] - df['Initial_MD'] / df['Days_Since_Report_to_True_Offer'] * df['Days_Since_Report_to_Date_Offer']

    # Computing 1-year BPV forecast for the bond
    df['BPV'] = (df['Market_Price'] / 100 * df['Nominal'] * df['Quantity'] + df['AI']) * df['MD_on_Date'] / 100

    # Drop rows with empty BPV
    df.dropna(subset = ['BPV'], inplace = True)    
    
    appended = appended.append(df)

appended_final = appended[['DATE', 'BOOK', 'Instrument_Name', 'Nearest_Offer', 'TRUE_Nearest_Offer', 
                           'MD_on_Date', 'Market_Price', 'Nominal', 'Quantity', 'Nearest_Coupon', 
                           'TRUE_Nearest_Coupon', 'AI_0', 'AI_One_Day', 'AI', 'BPV']]

# Save to csv appended file
os.chdir('M:/SHARE/Risk_Invest/Рыночные риски/Исходные файлы для отчетов/Monthly BPV Report')
if os.path.exists('BPV_APPENDED.xlsx'):
    os.remove('BPV_APPENDED.xlsx')

appended_final.to_excel('M:/SHARE/Risk_Invest/Рыночные риски/Исходные файлы для отчетов/Monthly BPV Report/BPV_APPENDED.xlsx', index = False, encoding = 'cp1251')